# Ponteiros em C — Tutorial

**Programação C (COMP0512) — UFS — 2026.2**

Este tutorial retoma o programa `troca_soma.c` visto em aula e continua de onde a aula parou:
as **tarefas de modificação** e o **desafio**. Ao final há exercícios de prática sobre cada tópico.

## Objetivos

Ao final deste tutorial você será capaz de:

- Explicar o que um ponteiro guarda e por que a memória tem endereços;
- Usar `&` (endereço de) e `*` (conteúdo de) corretamente, inclusive em declarações;
- Escrever funções que alteram variáveis do chamador e que devolvem mais de um resultado;
- Usar aritmética de ponteiros para percorrer vetores e calcular posições;
- Reconhecer as armadilhas mais comuns (ponteiro não inicializado, `NULL`, `sizeof` de vetor x ponteiro).
- Ler e usar um ponteiro para ponteiro (`int **pp`), no material extra ao final.

> **Como usar:** cada célula `%%writefile` grava um arquivo `.c` no diretório do notebook;
> a célula seguinte compila com `gcc` e executa. Edite o código dentro da célula e execute de novo.

In [ ]:
# Verifique se o gcc está disponível no seu ambiente
!gcc --version | head -1

## O programa da aula

Antes de modificar qualquer coisa, recompile e execute o programa condutor **exatamente como saiu
da aula** e confira que a saída bate com a que vimos no slide.

In [ ]:
%%writefile troca_soma.c
#include <stdio.h>

void troca_v1(int a, int b) {
    int t = a;
    a = b;
    b = t;
}

void troca_v2(int *a, int *b) {
    int t = *a;
    *a = *b;
    *b = t;
}

int main(void) {
    int x = 10, y = 20;
    troca_v1(x, y);
    printf("v1: x=%d y=%d\n", x, y);
    troca_v2(&x, &y);
    printf("v2: x=%d y=%d\n", x, y);

    int v[5] = {2, 4, 6, 8, 10};
    int *p = v;
    printf("v0=%d *p=%d p2=%d\n", v[0], *p, *(p + 2));
    int soma = 0;
    for (int *q = v; q < v + 5; q++)
        soma += *q;
    printf("soma=%d\n", soma);
    return 0;
}

In [ ]:
!gcc -Wall troca_soma.c -o troca_soma && ./troca_soma
!./troca_soma | grep -q 'v2: x=20 y=10' && echo OK || echo 'Verifique: esperava v2: x=20 y=10'

## 1. O que um ponteiro guarda

Toda variável ocupa bytes da memória, e esse pedaço de memória tem um **endereço** — um número que
identifica sua posição. Um **ponteiro** é uma variável cujo valor é o endereço de outra variável.

O programa abaixo imprime valores e endereços lado a lado. Os endereços mudam a cada execução
(e isso é normal): o que importa é a **relação** entre eles — `p` vale exatamente o mesmo que `&x`.

In [ ]:
%%writefile enderecos.c
#include <stdio.h>

int main(void) {
    int  x = 10;
    int *p = &x;

    printf("valor de x   : %d\n", x);
    printf("endereco de x: %p\n", (void *) &x);
    printf("valor de p   : %p\n", (void *) p);    /* igual a &x */
    printf("conteudo *p  : %d\n", *p);
    printf("endereco de p: %p\n", (void *) &p);

    *p = 99;                       /* escrever em *p e escrever em x */
    printf("agora x vale %d\n", x);

    printf("sizeof(int)=%zu sizeof(int *)=%zu sizeof(double *)=%zu\n",
           sizeof(int), sizeof(int *), sizeof(double *));
    return 0;
}

In [ ]:
!gcc -Wall enderecos.c -o enderecos && ./enderecos
!./enderecos | grep -q 'agora x vale 99' && echo OK || echo 'Verifique: *p = 99 deveria alterar x'

### Pergunte-se

1. Por que `p` e `&x` imprimem o mesmo número, mas `&p` imprime outro?
2. `sizeof(int *)` e `sizeof(double *)` deram o mesmo valor. Por quê?
3. O que aconteceria se a linha `int *p = &x;` virasse apenas `int *p;` e ainda assim fizéssemos `*p = 99;`?
   (Não execute — responda primeiro; se quiser testar, faça em uma cópia do arquivo.)

## 2. Os operadores `&` e `*`

- `&x` — *endereço de* `x`;
- `*p` — *conteúdo apontado por* `p`.

São inversos: `*(&x)` é `x`. Cuidado com o contexto: em `int *p;` o `*` faz parte da **declaração**
("p é um ponteiro"); em `*p = 7;` o `*` é o **uso** ("vá até lá e escreva").

É por isso que `troca_v1` falhou e `troca_v2` funcionou: em C os argumentos são sempre copiados,
então só chegamos à variável original se copiarmos o **endereço** dela.

In [ ]:
%%writefile passagem.c
#include <stdio.h>

void incrementa_valor(int n)  { n = n + 1; }      /* mexe na copia  */
void incrementa_end(int *n)   { *n = *n + 1; }    /* mexe no original */

int main(void) {
    int c = 5;
    incrementa_valor(c);
    printf("depois de incrementa_valor: %d\n", c);
    incrementa_end(&c);
    printf("depois de incrementa_end  : %d\n", c);
    return 0;
}

In [ ]:
!gcc -Wall passagem.c -o passagem && ./passagem
!./passagem | grep -q 'incrementa_end  : 6' && echo OK || echo 'Verifique: esperava 6 apos incrementa_end'

## 3. Aritmética de ponteiros

A regra: **`p + k` avança `k * sizeof(tipo apontado)` bytes**. A aritmética conta *elementos*,
não bytes — por isso `*(p + 2)` deu 6 (e não 4) no programa da aula.

O experimento abaixo mostra o salto para três tipos diferentes.

In [ ]:
%%writefile aritmetica.c
#include <stdio.h>

int main(void) {
    int    vi[3];
    double vd[3];
    char   vc[3];

    printf("int   : salto de %ld bytes (sizeof=%zu)\n",
           (long)((char *)(vi + 1) - (char *) vi), sizeof(int));
    printf("double: salto de %ld bytes (sizeof=%zu)\n",
           (long)((char *)(vd + 1) - (char *) vd), sizeof(double));
    printf("char  : salto de %ld bytes (sizeof=%zu)\n",
           (long)((char *)(vc + 1) - (char *) vc), sizeof(char));

    int v[5] = {2, 4, 6, 8, 10};
    printf("v[2]=%d  *(v+2)=%d  diferenca &v[4]-v = %ld\n",
           v[2], *(v + 2), (long)(&v[4] - v));

    int *p = v;
    printf("sizeof(v)=%zu  sizeof(p)=%zu\n", sizeof(v), sizeof(p));
    return 0;
}

In [ ]:
!gcc -Wall aritmetica.c -o aritmetica && ./aritmetica
!./aritmetica | grep -q 'diferenca &v\[4\]-v = 4' && echo OK || echo 'Verifique a diferenca de ponteiros'

### Visualizando o vetor na memória

A figura abaixo (gerada em Python, só para visualização) mostra por que `p + 2` chega ao terceiro
elemento: cada `int` ocupa 4 bytes, então o endereço avança de 4 em 4.

In [ ]:
import matplotlib.pyplot as plt

valores = [2, 4, 6, 8, 10]
base = 1000
enderecos = [base + 4 * i for i in range(len(valores))]

fig, ax = plt.subplots(figsize=(9, 2.2))
for i, (val, end) in enumerate(zip(valores, enderecos)):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, fill=True, facecolor="#e8f0fb", edgecolor="#034ea2"))
    ax.text(i + 0.5, 0.55, str(val), ha="center", va="center", fontsize=13, family="monospace")
    ax.text(i + 0.5, 0.18, f"v[{i}]", ha="center", va="center", fontsize=8, color="#646464")
    ax.text(i + 0.5, -0.25, str(end), ha="center", va="center", fontsize=8, color="#646464")

ax.annotate("p", xy=(0.5, 1.0), xytext=(0.5, 1.7), ha="center", fontsize=11,
            arrowprops=dict(arrowstyle="->", color="#034ea2", lw=1.5))
ax.annotate("p + 2", xy=(2.5, 1.0), xytext=(2.5, 1.7), ha="center", fontsize=11, color="#ed1c24",
            arrowprops=dict(arrowstyle="->", color="#ed1c24", lw=1.5))

ax.set_xlim(-0.3, len(valores) + 0.3)
ax.set_ylim(-0.6, 2.1)
ax.axis("off")
plt.title("p + k avança k × sizeof(int) = 4k bytes", fontsize=11)
plt.show()

## Altere o programa

As três tarefas da aula, em ordem crescente de desafio. Para cada uma: **preveja a saída antes de
executar**, depois compile e compare.

### Tarefa 1 — dobrar sem colchetes

Complete `dobra` para multiplicar cada elemento por 2 usando **apenas** `*` e aritmética de
ponteiros — sem `[ ]` em nenhum lugar da função.

In [ ]:
%%writefile tarefa1.c
#include <stdio.h>

/* TODO: percorra o vetor com um ponteiro e dobre cada elemento.
   Restricao: nao use colchetes dentro desta funcao.          */
void dobra(int *v, int n) {
    /* implemente aqui */
}

int main(void) {
    int v[5] = {2, 4, 6, 8, 10};
    dobra(v, 5);
    for (int i = 0; i < 5; i++)
        printf("%d ", v[i]);
    printf("\n");
    return 0;
}

In [ ]:
# Teste da Tarefa 1: esperado "4 8 12 16 20"
!gcc -Wall tarefa1.c -o tarefa1
!./tarefa1 | grep -q '4 8 12 16 20' && echo OK || echo 'Verifique: esperava 4 8 12 16 20'
print('Lembre da restricao: nenhum colchete dentro de dobra().')

### Tarefa 2 — devolver um endereço

Implemente `int *maior(int *v, int n)` que devolve o **endereço** do maior elemento. No `main`,
imprima o valor com `*m` e a posição com `m - v`.

In [ ]:
%%writefile tarefa2.c
#include <stdio.h>

/* TODO: devolva o ENDERECO do maior elemento (nao o valor). */
int *maior(int *v, int n) {
    int *m = v;
    /* implemente aqui */
    return m;
}

int main(void) {
    int v[5] = {2, 9, 6, 8, 10};
    int *m = maior(v, 5);
    printf("maior=%d posicao=%ld\n", *m, (long)(m - v));
    return 0;
}

In [ ]:
# Teste da Tarefa 2: esperado "maior=10 posicao=4"
!gcc -Wall tarefa2.c -o tarefa2
!./tarefa2 | grep -q 'maior=10 posicao=4' && echo OK || echo 'Verifique: esperava maior=10 posicao=4'

### Tarefa 3 — dois resultados de uma vez

Implemente `void min_max(int *v, int n, int *pmin, int *pmax)`. A função não devolve nada com
`return`: ela **escreve** nas duas variáveis do `main` cujos endereços recebeu.

In [ ]:
%%writefile tarefa3.c
#include <stdio.h>

/* TODO: escreva o menor em *pmin e o maior em *pmax. */
void min_max(int *v, int n, int *pmin, int *pmax) {
    /* implemente aqui */
}

int main(void) {
    int v[6] = {7, 2, 9, 4, 10, 3};
    int menor = 0, maior = 0;
    min_max(v, 6, &menor, &maior);
    printf("min=%d max=%d\n", menor, maior);
    return 0;
}

In [ ]:
# Teste da Tarefa 3: esperado "min=2 max=10"
!gcc -Wall tarefa3.c -o tarefa3
!./tarefa3 | grep -q 'min=2 max=10' && echo OK || echo 'Verifique: esperava min=2 max=10'

## Desafio — inverter um vetor com dois ponteiros

Escreva `void inverte(int *ini, int *fim)`, onde `ini` é o endereço do primeiro elemento e `fim` é o
endereço **logo após** o último (a chamada é `inverte(v, v + n)`). Inverta o vetor no lugar:
troque os extremos e caminhe para o centro, **sem índices e sem vetor auxiliar**.

Critérios: funcionar para tamanho par e ímpar; reaproveitar `troca_v2` para a troca.

In [ ]:
%%writefile desafio.c
#include <stdio.h>

void troca_v2(int *a, int *b) {
    int t = *a;
    *a = *b;
    *b = t;
}

/* TODO: fim aponta para UMA posicao depois do ultimo elemento. */
void inverte(int *ini, int *fim) {
    /* implemente aqui */
}

void imprime(int *v, int n) {
    for (int *q = v; q < v + n; q++)
        printf("%d ", *q);
    printf("\n");
}

int main(void) {
    int a[5] = {1, 2, 3, 4, 5};
    int b[6] = {1, 2, 3, 4, 5, 6};
    inverte(a, a + 5);
    inverte(b, b + 6);
    imprime(a, 5);
    imprime(b, 6);
    return 0;
}

In [ ]:
# Teste do desafio: esperado "5 4 3 2 1" e "6 5 4 3 2 1"
!gcc -Wall desafio.c -o desafio
!./desafio | head -1 | grep -q '5 4 3 2 1' && echo 'OK (impar)' || echo 'Verifique o caso impar'
!./desafio | tail -1 | grep -q '6 5 4 3 2 1' && echo 'OK (par)' || echo 'Verifique o caso par'

## Extra — ponteiros para ponteiros

Um ponteiro também é uma variável: ocupa memória e tem endereço (`&p`). Nada impede que **outra**
variável guarde esse endereço — é o que faz `int **pp = &p;`.

Cada `*` a mais na declaração é um nível de indireção; cada `*` no uso dá um passo até o valor:

| Expressão | Tipo | Vale |
|---|---|---|
| `pp` | `int **` | o endereço de `p` |
| `*pp` | `int *` | o próprio `p`, ou seja, o endereço de `x` |
| `**pp` | `int` | o próprio `x` |

**Preveja antes de executar:** o que a célula abaixo imprime nas linhas de `**pp` e de `x` depois
de `**pp = 99;`?


In [ ]:
%%writefile ponteiro_ponteiro.c
#include <stdio.h>

/* Para alterar um int, a funcao recebe int *.
   Para alterar um int *, ela recebe int **.   */
void aponta_maior(int *v, int n, int **saida) {
    int *m = v;
    for (int i = 1; i < n; i++)
        if (v[i] > *m) m = &v[i];
    *saida = m;              /* escreve no ponteiro do chamador */
}

int main(void) {
    int   x  = 10;
    int  *p  = &x;    /* p  guarda o endereco de x */
    int **pp = &p;    /* pp guarda o endereco de p */

    printf("x    = %d\n", x);
    printf("*p   = %d\n", *p);
    printf("**pp = %d\n", **pp);

    printf("p    = %p  (endereco de x)\n", (void *) p);
    printf("*pp  = %p  (o mesmo: *pp E o proprio p)\n", (void *) *pp);
    printf("pp   = %p  (endereco de p)\n", (void *) pp);

    **pp = 99;
    printf("depois de **pp = 99, x vale %d\n", x);

    int v[5] = {2, 4, 10, 8, 6};
    int *achou = NULL;
    aponta_maior(v, 5, &achou);
    printf("maior=%d posicao=%ld\n", *achou, achou - v);

    return 0;
}

In [ ]:
!gcc -Wall ponteiro_ponteiro.c -o ponteiro_ponteiro && ./ponteiro_ponteiro
!./ponteiro_ponteiro | grep -q 'x vale 99' && echo OK || echo 'Verifique: **pp = 99 deveria alterar x'
!./ponteiro_ponteiro | grep -q 'maior=10 posicao=2' && echo OK || echo 'Verifique aponta_maior'

### Pergunte-se

1. Por que `p` e `*pp` imprimem o mesmo número, mas `pp` imprime outro?
2. `*pp` **não** é o valor de `x` — é um endereço. Qual erro o compilador acusa em `printf("%d", *pp);`?
3. Em `aponta_maior`, o que aconteceria se o parâmetro fosse `int *saida` e a atribuição fosse
   `saida = m;`? Por que `achou` continuaria `NULL`?
4. `main(int argc, char **argv)` usa a mesma ideia. O que `argv`, `*argv` e `**argv` representam?

**Para praticar:** escreva `void zera_ponteiro(int **pp)` que faz o ponteiro do chamador virar `NULL`,
e mostre no `main` que, depois da chamada, `if (achou == NULL)` é verdadeiro.


## Exercícios de prática

**Conceituais** (responda sem computador, depois confira executando):

1. Se `int x = 7; int *p = &x; int **pp = &p;` — quanto valem `*p`, `**pp` e `*pp`?
2. Por que `int *a, b;` **não** declara dois ponteiros? Como declarar dois?
3. Dado `int v[10]; int *p = v;`, qual a diferença entre `sizeof(v)` e `sizeof(p)`? Por quê?
4. `p++` e `(*p)++` fazem coisas diferentes. Explique cada uma.
5. Por que `v + 5` (com `int v[5]`) pode ser calculado e comparado, mas nunca desreferenciado?

**De programação:**

6. Escreva `int conta_pares(const int *ini, const int *fim)` que conta quantos elementos pares
   existem no intervalo `[ini, fim)`, usando apenas ponteiros.
7. Escreva `void copia(const int *origem, int *destino, int n)` sem usar colchetes.
8. Escreva `int igual(const int *a, const int *b, int n)` que devolve 1 se os dois vetores forem
   iguais elemento a elemento e 0 caso contrário — parando na primeira diferença.
9. Reescreva a função `media` da primeira aula usando aritmética de ponteiros no lugar do índice.
10. Escreva `int *busca(int *v, int n, int alvo)` que devolve o endereço da primeira ocorrência de
    `alvo`, ou `NULL` se não houver. No `main`, teste o retorno com `if (r != NULL)`.

In [ ]:
%%writefile pratica.c
#include <stdio.h>

/* Espaco livre para os exercicios 6 a 10. */

int main(void) {
    printf("escreva seus testes aqui\n");
    return 0;
}

In [ ]:
!gcc -Wall pratica.c -o pratica && ./pratica

## Referências

- KERNIGHAN, B. W.; RITCHIE, D. M. *C: a linguagem de programação — padrão ANSI*. Campus, 1989 (cap. 5).
- SCHILDT, H. *C completo e total*. 3. ed. Makron Books, 1997.
- TENENBAUM, A. M.; LANGSAM, Y.; AUGENSTEIN, M. J. *Estruturas de dados usando C*. Pearson Makron Books, 1995.

Lista completa em `../referencias.bib`.